In [2]:
import pandas as pd
import numpy as np
import regex as re
import os, sys, gzip, pyarrow.parquet

Local_google_drive = "/Users/siavash/Library/CloudStorage/GoogleDrive-siavash.jafarizadeh@gmail.com/.shortcut-targets-by-id/1BA94HYNI6NLWOcU5Ts7Nfso4uuVjrDwp/deeplearning2026"

df = pd.read_parquet('FNDDSeverything.parquet.gzip')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 480176 entries, 0 to 480175
Data columns (total 92 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   description       480176 non-null  str    
 1   ALA_G             2408 non-null    float64
 2   ARA_G             154 non-null     float64
 3   B12_UG            22124 non-null   float64
 4   B1_MG             25807 non-null   float64
 5   B2_MG             38109 non-null   float64
 6   B3_MG             41473 non-null   float64
 7   B5_MG             13908 non-null   float64
 8   B6_MG             31844 non-null   float64
 9   B7_UG             1348 non-null    float64
 10  B9_UG             26260 non-null   float64
 11  B_UG              822 non-null     float64
 12  CHO1_G            436060 non-null  float64
 13  CHOfiber1_G       270661 non-null  float64
 14  CHOstarch_G       1975 non-null    float64
 15  CHOsugar_G        696 non-null     float64
 16  Ca_MG             303735 non-nu

In [4]:
df_str = df.select_dtypes(include="str")

print(df_str.head)
# print()
# print(df_str.info())

<bound method NDFrame.head of nameunit                                        description  \
0          ADVANCEPIERRE FLAMEBROILED RIB SHAPED PORK PA...   
1                   ALL NATURAL GLUTEN FREE CHICKEN NUGGETS   
2          ALL NATURAL ROSEMARY & OLIVE OIL BASMATI RICE...   
3                                               ALMOND MILK   
4                                     ALMOND MILK, ORIGINAL   
...                                                     ...   
480171    [TROPICAL BOOST] GUARANA + CATUABA ADAPTOGEN-P...   
480172                                       \ROUND CHALLAH   
480173    \WALNUT CHOCOLATE CHIP COOKIES, WALNUT CHOCOLA...   
480174       {GO BOLD} SPICY CILANTRO SAUCE, SPICY CILANTRO   
480175                                          {TRAIL MIX}   

nameunit                                         category     data_type  
0         Meat/Poultry/Other Animals - Prepared/Processed  branded_food  
1                        Frozen Poultry, Chicken & Turkey  brande

In [5]:
for i in [24, 89,124,134, 550, 551, 552, 553]:
    print(f" the description at {i} is\n\t {df_str.description[i]}\n the category is \n {df_str.category[i]}")
    print('***')

 the description at 24 is
	  DEATH WISH COFFEE CO LATTE, 8 FL OZ
 the category is 
 Other Drinks
***
 the description at 89 is
	  PUMPKIN CARAMEL APPLE MILK CHOLOCOLATE, PUMPKIN; CARAMEL APPLE
 the category is 
 Chocolate
***
 the description at 124 is
	  ZERO CALORIE SPORTS DRINK, FRUIT PUNCH
 the category is 
 Sport Drinks
***
 the description at 134 is
	 ""BACN,CAND,FL,4PC, Z""
 the category is 
 Meat/Poultry/Other Animals – Prepared/Processed
***
 the description at 550 is
	 .38 SPECIAL GOOD ON EVERYTHING SEASONING, .38 SPECIAL
 the category is 
 Seasoning Mixes, Salts, Marinades & Tenderizers
***
 the description at 551 is
	 .5-.75 WGGS LANE SNAPPER
 the category is 
 Fish  Unprepared/Unprocessed
***
 the description at 552 is
	 .5-1 LB IPW CLEAN OCTOPUS
 the category is 
 nan
***
 the description at 553 is
	 .5/1 LB W.G.G.S. SNAPPER
 the category is 
 Fish  Unprepared/Unprocessed
***


Some people, when confronted with a problem, think
“I know, I'll use regular expressions.”   Now they have two problems.[Link](https://regex.info/blog/2006-09-|/247) 

For processing text, I have used **regex** which is almost the same as pyhon's **re** but with slightly better coverage.

For documentations look at [Regular Expression howto](https://docs.python.org/3/howto/regex.html#regex-howto) and [Python for linguists](https://v4py.github.io/regex.html#:~:text=Unlike%20in%20Perl%20or%20Ruby,without%20needing%20to%20install%20anything). 


In [161]:
Pattern = re.compile(r'\W|_')

for c in df_str.columns:
    ds = df_str[c].fillna('-')
    l = []
    S = []
    for index in range(len(ds)):
        match = Pattern.findall(ds[index])
        l.extend(match)
    S = sorted(set(l))
    print(f"At the column {c}, the special chars are:\n", S)
    print("with length =", len(S))


At the column description, the special chars are:
 ['\t', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '_', '`', '{', '|', '}', '~', '\xa0']
with length = 34
At the column category, the special chars are:
 [' ', '&', "'", '(', ')', ',', '-', '.', '/', ':', '–']
with length = 11
At the column data_type, the special chars are:
 ['-', '_']
with length = 2


A better and less memory usage method below.

In [12]:
Pattern = re.compile(r'\W|_')

for c in df_str.columns:
    Col = df_str[c].fillna('-')
    col_str = Col.astype(str)
    match = Pattern.findall(''.join(col_str))  # Find all matches across the entire column at once

    S = sorted(set(match))
    print(f"At the column {c}, the special chars are:\n", S)
    print(len(S))

At the column description, the special chars are:
 ['\t', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '_', '`', '{', '|', '}', '~', '\xa0']
34
At the column category, the special chars are:
 [' ', '&', "'", '(', ')', ',', '-', '.', '/', ':', '–']
11
At the column data_type, the special chars are:
 ['-', '_']
2


### Lets remove the special characters using python itself.

In [ ]:
import string

### Option |for removing special char without use of regex.

text = df_str.description[480171]

print(text)

translator = str.maketrans('', '', string.punctuation)
cleaned_text_translate = text.translate(translator)
print(cleaned_text_translate)

### Option 2 for removing special char. The difference is tha we can choose to replace '-' with ' '.

translator_dic={}
for char in string.punctuation:
    translator_dic[char] = ''

# print(translator_dic)

translator_2 = str.maketrans(translator_dic)
cleaned_text_strip = text.translate(translator_2)
print(cleaned_text_strip)

[TROPICAL BOOST] GUARANA + CATUABA ADAPTOGEN-POWERED BEVERAGE, TROPICAL BOOST GUARANA + CATUABA
TROPICAL BOOST GUARANA  CATUABA ADAPTOGENPOWERED BEVERAGE TROPICAL BOOST GUARANA  CATUABA
32
{'!': '', '"': '', '#': '', '$': '', '%': '', '&': '', "'": '', '(': '', ')': '', '*': '', '+': '', ',': '', '-': '', '.': '', '/': '', ':': '', ';': '', '<': '', '=': '', '>': '', '?': '', '@': '', '[': '', '\\': '', ']': '', '^': '', '_': '', '`': '', '{': '', '|': '', '}': '', '~': ''}
TROPICAL BOOST GUARANA  CATUABA ADAPTOGENPOWERED BEVERAGE TROPICAL BOOST GUARANA  CATUABA


POS tag list:
| Abbreviation | meaning |
|---|---|
| CC|coordinating conjunction|
| CD|cardinal digit|
| DT|determiner|
| EX|existential there (like: "there is" ... think of it like "there exists")|
| FW|foreign word|
| IN|preposition/subordinating conjunction|
| JJ|adjective	'big'|
| JJR|adjective, comparative	'bigger'|
| JJS|adjective, superlative	'biggest'|
| LS|list marker|
| MD|modal	could, will|
| NN|noun, singular 'desk'|
| NNS|noun plural	'desks'|
| NNP|proper noun, singular	'Harrison'|
| NNPS|proper noun, plural	'Americans'|
| PDT|predeterminer	'all the kids'|
| POS|possessive ending	parent's|
| PRP|personal pronoun	I, he, she|
| PRP$|possessive pronoun	my, his, hers|
| RB|adverb	very, silently,|
| RBR|adverb, comparative	better|
| RBS|adverb, superlative	best|
| RP|particle	give up|
| TO|to	go 'to' the store.|
| UH|interjection	errrrrrrrm|
| VB|verb, base form	take|
| VBD|verb, past tense	took|
| VBG|verb, gerund/present participle	taking|
| VBN|verb, past participle	taken|
| VBP|verb, sing. present, non-3d	take|
| VBZ|verb, 3rd person sing. present	takes|
| WDT|wh-determiner	which|
| WP|wh-pronoun	who, what|
| WP$|possessive wh-pronoun	whose|
| WRB|wh-abverb	where, when|

In [31]:
from nltk.tokenize import RegexpTokenizer

tokenizer = RegexpTokenizer(r'\w+')

text = df_str.description[480171]
tokens = tokenizer.tokenize(text)
print(text)
print(tokens)

[TROPICAL BOOST] GUARANA + CATUABA ADAPTOGEN-POWERED BEVERAGE, TROPICAL BOOST GUARANA + CATUABA
['TROPICAL', 'BOOST', 'GUARANA', 'CATUABA', 'ADAPTOGEN', 'POWERED', 'BEVERAGE', 'TROPICAL', 'BOOST', 'GUARANA', 'CATUABA']


In [ ]:
import nltk
# nltk.download('punkt_tab')
# nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize
from nltk import pos_tag

text = df_str.description[480171]

tokens = tokenizer.tokenize(text)
tagged_tokens = pos_tag(tokens)

print(tagged_tokens)



[('TROPICAL', 'NNP'), ('BOOST', 'NNP'), ('GUARANA', 'NNP'), ('CATUABA', 'NNP'), ('ADAPTOGEN', 'NNP'), ('POWERED', 'NNP'), ('BEVERAGE', 'NNP'), ('TROPICAL', 'NNP'), ('BOOST', 'NNP'), ('GUARANA', 'NNP'), ('CATUABA', 'NNP')]


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Example dataframe - replace with your actual data
df = pd.DataFrame({
    'description': [
        'fresh organic tomatoes red',
        'sweet ripe Banana yellow',
        'crunchy green apple',
        'juicy red ,strawberries',
        # ... your actual food descriptions
    ]
})

# Preprocess text
df['description_clean'] = df['description'].str.lower().str.strip()
print(df)

tfidf = TfidfVectorizer(
    max_features=2000,  # adjust based on your vocabulary size
    ngram_range=(1, 2),  # captures single words and word pairs
    min_df=1
)
X_tfidf = tfidf.fit_transform(df['description_clean']).toarray()
print(X_tfidf)

                  description           description_clean
0  fresh organic tomatoes red  fresh organic tomatoes red
1    sweet ripe Banana yellow    sweet ripe banana yellow
2         crunchy green apple         crunchy green apple
3     juicy red ,strawberries     juicy red ,strawberries
[[0.         0.         0.         0.         0.         0.38861429
  0.38861429 0.         0.         0.         0.         0.38861429
  0.38861429 0.30638797 0.         0.         0.         0.
  0.         0.         0.38861429 0.38861429 0.        ]
 [0.         0.37796447 0.37796447 0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.37796447 0.37796447 0.
  0.37796447 0.37796447 0.         0.         0.37796447]
 [0.4472136  0.         0.         0.4472136  0.4472136  0.
  0.         0.4472136  0.4472136  0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0. 

In [21]:
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim

# Download required NLTK data. Uncomment and download for the first run. Then comment to space.
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')



translator = str.maketrans('', '', string.punctuation)

df_description = df_str.description

df_clean=[]

for text in df_description:
    clean_text_translate = text.translate(translator)
    df_clean.append(clean_text_translate)

print(len(df_clean))

tfidf = TfidfVectorizer(
    max_features=2000,  # adjust based on your vocabulary size
    ngram_range=(1, 2),  # captures single words and word pairs
    min_df=1
)
X_tfidf = tfidf.fit_transform(df_clean).toarray()

# Convert to PyTorch tensors
X_tensor_tfidf = torch.FloatTensor(X_tfidf)

# Define autoencoder for clustering
input_dim = X_tfidf.shape[1]
encoding_dim = 50  # bottleneck dimension for clustering
n_clusters = 20  # adjust to your expected number of food categories

# Build autoencoder
encoder_tfidf = nn.Sequential(
    nn.Linear(input_dim, 64),
    nn.ReLU(),
    nn.Linear(64, encoding_dim),
    nn.ReLU()
)

decoder_tfidf = nn.Sequential(
    nn.Linear(encoding_dim, 64),
    nn.ReLU(),
    nn.Linear(64, input_dim),
    nn.Sigmoid()
)

clustering_layer_tfidf = nn.Linear(encoding_dim, n_clusters)

# Training setup
criterion = nn.MSELoss()
optimizer = optim.Adam(
    list(encoder_tfidf.parameters()) + 
    list(decoder_tfidf.parameters()) + 
    list(clustering_layer_tfidf.parameters()), 
    lr=0.001
)

# Training loop with more epochs
n_epochs = 50  # train longer for better clustering

for epoch in range(n_epochs):
    # Forward pass
    encoded = encoder_tfidf(X_tensor_tfidf)
    decoded = decoder_tfidf(encoded)
    cluster_logits = clustering_layer_tfidf(encoded)
    
    # Loss: reconstruction + clustering (pseudo-labels via argmax)
    reconstruction_loss = criterion(decoded, X_tensor_tfidf)
    
    # Clustering loss using KL divergence with soft assignments
    cluster_probs = torch.softmax(cluster_logits, dim=1)
    target_dist = cluster_probs ** 2 / cluster_probs.sum(0)
    target_dist = target_dist / target_dist.sum(1, keepdim=True)
    clustering_loss = -(target_dist * torch.log(cluster_probs + 1e-10)).sum(1).mean()
    
    total_loss = reconstruction_loss + 0.1 * clustering_loss
    
    # Backward pass
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {total_loss.item():.4f}')
'''

# Get cluster assignments
with torch.no_grad():
    encoded_tfidf = encoder_tfidf(X_tensor_tfidf)
    cluster_assignments_tfidf = torch.argmax(clustering_layer_tfidf(encoded_tfidf), dim=1)

df['cluster_tfidf'] = cluster_assignments_tfidf.numpy()
print("\nTF-IDF Clustering Results:")
print(df[['description', 'cluster_tfidf']])

# ===== METHOD 2: Count Vectorization (Simpler, frequency-based) =====
print("\n\nMETHOD 2: Count Vectorization")

count_vec = CountVectorizer(
    max_features=100,
    ngram_range=(1, 2),
    min_df=1
)
X_count = count_vec.fit_transform(df['description_clean']).toarray()

# Normalize counts
scaler = StandardScaler()
X_count_scaled = scaler.fit_transform(X_count)
X_tensor_count = torch.FloatTensor(X_count_scaled)

# Build new autoencoder for count vectors
input_dim_count = X_count_scaled.shape[1]

encoder_count = nn.Sequential(
    nn.Linear(input_dim_count, 64),
    nn.ReLU(),
    nn.Linear(64, encoding_dim),
    nn.ReLU()
)

decoder_count = nn.Sequential(
    nn.Linear(encoding_dim, 64),
    nn.ReLU(),
    nn.Linear(64, input_dim_count),
    nn.Tanh()
)

clustering_layer_count = nn.Linear(encoding_dim, n_clusters)

optimizer_count = optim.Adam(
    list(encoder_count.parameters()) + 
    list(decoder_count.parameters()) + 
    list(clustering_layer_count.parameters()), 
    lr=0.001
)

# Training loop
for epoch in range(n_epochs):
    encoded = encoder_count(X_tensor_count)
    decoded = decoder_count(encoded)
    cluster_logits = clustering_layer_count(encoded)
    
    reconstruction_loss = criterion(decoded, X_tensor_count)
    
    cluster_probs = torch.softmax(cluster_logits, dim=1)
    target_dist = cluster_probs ** 2 / cluster_probs.sum(0)
    target_dist = target_dist / target_dist.sum(1, keepdim=True)
    clustering_loss = -(target_dist * torch.log(cluster_probs + 1e-10)).sum(1).mean()
    
    total_loss = reconstruction_loss + 0.1 * clustering_loss
    
    optimizer_count.zero_grad()
    total_loss.backward()
    optimizer_count.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {total_loss.item():.4f}')

# Get cluster assignments
with torch.no_grad():
    encoded_count = encoder_count(X_tensor_count)
    cluster_assignments_count = torch.argmax(clustering_layer_count(encoded_count), dim=1)

df['cluster_count'] = cluster_assignments_count.numpy()
print("\nCount Vectorization Clustering Results:")
print(df[['description', 'cluster_count']])

# ===== METHOD 3: Binary (Presence/Absence) Vectorization =====
print("\n\nMETHOD 3: Binary Vectorization")

binary_vec = CountVectorizer(
    max_features=100,
    binary=True,  # just presence/absence
    ngram_range=(1, 1),  # single words work well for nouns
    min_df=1
)
X_binary = binary_vec.fit_transform(df['description_clean']).toarray()
X_tensor_binary = torch.FloatTensor(X_binary)

# This method is fastest and works well for noun-heavy descriptions
# Build simpler network since binary features are sparse
encoder_binary = nn.Sequential(
    nn.Linear(X_binary.shape[1], 32),
    nn.ReLU(),
    nn.Linear(32, encoding_dim),
    nn.ReLU()
)

clustering_layer_binary = nn.Linear(encoding_dim, n_clusters)

optimizer_binary = optim.Adam(
    list(encoder_binary.parameters()) + 
    list(clustering_layer_binary.parameters()), 
    lr=0.001
)

# Training (no decoder needed, just clustering)
for epoch in range(n_epochs):
    encoded = encoder_binary(X_tensor_binary)
    cluster_logits = clustering_layer_binary(encoded)
    
    cluster_probs = torch.softmax(cluster_logits, dim=1)
    target_dist = cluster_probs ** 2 / cluster_probs.sum(0)
    target_dist = target_dist / target_dist.sum(1, keepdim=True)
    clustering_loss = -(target_dist * torch.log(cluster_probs + 1e-10)).sum(1, keepdim=True).mean()
    
    optimizer_binary.zero_grad()
    clustering_loss.backward()
    optimizer_binary.step()
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {clustering_loss.item():.4f}')

with torch.no_grad():
    encoded_binary = encoder_binary(X_tensor_binary)
    cluster_assignments_binary = torch.argmax(clustering_layer_binary(encoded_binary), dim=1)

df['cluster_binary'] = cluster_assignments_binary.numpy()
print("\nBinary Vectorization Clustering Results:")
print(df[['description', 'cluster_binary']])

# Compare all methods
print("\n===== COMPARISON OF ALL METHODS =====")
print(df)
'''


480176
Epoch [10/50], Loss: 0.5392
Epoch [20/50], Loss: 0.5210
Epoch [30/50], Loss: 0.4813
Epoch [40/50], Loss: 0.3961
Epoch [50/50], Loss: 0.2940


'\n\n# Get cluster assignments\nwith torch.no_grad():\n    encoded_tfidf = encoder_tfidf(X_tensor_tfidf)\n    cluster_assignments_tfidf = torch.argmax(clustering_layer_tfidf(encoded_tfidf), dim=1)\n\ndf[\'cluster_tfidf\'] = cluster_assignments_tfidf.numpy()\nprint("\nTF-IDF Clustering Results:")\nprint(df[[\'description\', \'cluster_tfidf\']])\n\n# ===== METHOD 2: Count Vectorization (Simpler, frequency-based) =====\nprint("\n\nMETHOD 2: Count Vectorization")\n\ncount_vec = CountVectorizer(\n    max_features=100,\n    ngram_range=(1, 2),\n    min_df=1\n)\nX_count = count_vec.fit_transform(df[\'description_clean\']).toarray()\n\n# Normalize counts\nscaler = StandardScaler()\nX_count_scaled = scaler.fit_transform(X_count)\nX_tensor_count = torch.FloatTensor(X_count_scaled)\n\n# Build new autoencoder for count vectors\ninput_dim_count = X_count_scaled.shape[1]\n\nencoder_count = nn.Sequential(\n    nn.Linear(input_dim_count, 64),\n    nn.ReLU(),\n    nn.Linear(64, encoding_dim),\n    nn.R

In [ ]:
batch_size = 512
n_epochs = 50  # fewer epochs but with batches

# Convert to PyTorch dataset for batch processing
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X_tensor_tfidf)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Modified training loop with batches
for epoch in range(n_epochs):
    epoch_loss = 0
    for batch_idx, (batch_X,) in enumerate(dataloader):
        # Forward pass
        encoded = encoder_tfidf(batch_X)
        decoded = decoder_tfidf(encoded)
        cluster_logits = clustering_layer_tfidf(encoded)
        
        reconstruction_loss = criterion(decoded, batch_X)
        
        cluster_probs = torch.softmax(cluster_logits, dim=1)
        target_dist = cluster_probs ** 2 / cluster_probs.sum(0)
        target_dist = target_dist / target_dist.sum(1, keepdim=True)
        clustering_loss = -(target_dist * torch.log(cluster_probs + 1e-10)).sum(1).mean()
        
        total_loss = reconstruction_loss + 0.1 * clustering_loss
        
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        epoch_loss += total_loss.item()
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1}/{n_epochs}], Avg Loss: {epoch_loss/len(dataloader):.4f}')


## Memory-efficient vectorization settings:


In [ ]:

# For very large datasets, consider:
tfidf = TfidfVectorizer(
    max_features=1500,  # Balance between information and memory
    ngram_range=(1, 1),  # Start with unigrams only (faster)
    min_df=20,  # Ignore words in fewer than 20 documents
    max_df=0.8,  # Ignore words in more than 80% of documents (too common)
    stop_words='english'  # Remove common English words
)
